# Day 8: 公共データ GEO/SRA の読み方

## この章で解く現実の課題

GEOやSRAのmetadataから、炎症刺激RNA-seqとして使えるサンプルを選びます。

現実の解析では、公共データを再解析する機会が多くあります。しかし、アクセッション番号だけを見ても解析できません。細胞種、処理条件、処理時間、replicate、ライブラリタイプを読み、比較可能なサンプルを選ぶ必要があります。


## 最小限の用語

- GEO: 論文に紐づく遺伝子発現データの登録先。
- SRA: 生のシーケンスreadを保存するデータベース。
- GSE: GEOの研究全体。
- GSM: GEOの個々のサンプル。
- SRR: SRAのrun ID。FASTQ取得に使われることが多い。
- BioProject: 研究プロジェクト単位の登録。


In [ ]:
import pandas as pd

# 実際のGEO metadataを読む前の練習用ミニ表。
# 現実にはGEOのSeries Matrixや補足ファイルからこのような表を作る。
geo_samples = pd.DataFrame({
    "GSM": ["GSM0001", "GSM0002", "GSM0003", "GSM0004", "GSM0005", "GSM0006", "GSM0007"],
    "SRR": ["SRR1001", "SRR1002", "SRR1003", "SRR1004", "SRR1005", "SRR1006", "SRR1007"],
    "cell_type": ["macrophage", "macrophage", "macrophage", "macrophage", "fibroblast", "macrophage", "macrophage"],
    "condition": ["control", "control", "LPS", "LPS", "LPS", "LPS", "control"],
    "time_h": [6, 6, 6, 6, 6, 24, 6],
    "library_strategy": ["RNA-Seq"] * 7,
    "layout": ["PAIRED", "PAIRED", "PAIRED", "PAIRED", "PAIRED", "PAIRED", "SINGLE"],
})
geo_samples


## サンプル選択の考え方

今回のケーススタディでは、同じ細胞種、同じ時間、同じライブラリ条件で、controlとLPS刺激を比較したいとします。

そのため、以下のようなサンプルは除外候補です。

- 細胞種が違う: macrophageではなくfibroblast
- 処理時間が違う: 6時間ではなく24時間
- ライブラリ形式が違う: paired-endとsingle-endが混在
- replicateが不足している: 各条件1サンプルしかない


In [ ]:
selected = geo_samples.query(
    "cell_type == 'macrophage' and time_h == 6 and layout == 'PAIRED'"
).copy()
selected


In [ ]:
# 条件ごとのサンプル数を確認する。
# RNA-seqの比較では、各条件にbiological replicateが複数あることが重要。
selected["condition"].value_counts()


In [ ]:
def sample_selection_report(df):
    return pd.DataFrame({
        "criterion": ["same cell type", "same time point", "same layout", "replicates per condition"],
        "why_it_matters": [
            "細胞種が違うと発現差の大部分が細胞種差になる",
            "時間が違うと応答段階が変わる",
            "測定方式の違いが定量差を生む可能性がある",
            "ばらつきを見積もれないと信頼性が低い",
        ],
        "this_example": [
            selected["cell_type"].nunique() == 1,
            selected["time_h"].nunique() == 1,
            selected["layout"].nunique() == 1,
            selected["condition"].value_counts().min() >= 2,
        ],
    })

sample_selection_report(selected)


## 読み取り

公共データ解析の最初の成果物は、FASTQでもcount matrixでもなく、解析対象サンプルを定義したmetadata表です。

この表により、「何と何を比較しているのか」「除外したサンプルはなぜ除外したのか」を説明できます。これはGitHubポートフォリオでも重要です。

## エラーが出た場合

- `query`でエラー: 文字列条件の引用符が崩れている可能性があります。
- サンプル数が足りない: そのデータセット単独での差次的発現解析は弱いです。別データセットを探す判断も実務です。

## この章のゴール

GEO/SRAのmetadataから、比較可能なサンプルを選び、除外理由を説明できれば合格です。
